# 第 9 章 · Executor 与回调组：谁在什么时候、哪个线程里跑你的回调

本章回答：一个慢回调为什么会让别的回调全都卡住？多线程 executor 加上了为什么还是不并行？回调组是什么、怎么分组？第 5 章那个服务死锁到底怎么正确地修？

配套讲义：仓库里 `ros2/notes/lessons/09-executors.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 rclpy；而且真 Python 线程受 GIL 和系统调度影响，每次跑结果都不一样。这里把时间切成 1 毫秒一格，按讲义第 3 步的例子（50 Hz 控制定时器、30 Hz 图像订阅、图像回调 40 ms）
一格一格推进，结果每次都一样。规则：定时器晚了不补发，到点时上一拍还没跑就算漏拍；图像只留最新一张（Keep last 1）。

In [ ]:
from dataclasses import dataclass

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 3 步 单线程 | 一个线程、默认组，图像回调 40 ms | `simulate(single, threads=1)` | 控制跑 25 次、漏 23 拍 |
| 第 3 步 分组 | 控制一组、图像一组，两个线程 | `simulate(grouped, threads=2)` | 控制跑 49 次、漏 0 拍、晚 0 ms |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**类 `Callback`**：下面要用到的小工具。

In [ ]:
@dataclass
class Callback:
    name: str
    period_ms: float  # 多久来一次
    cost_ms: int  # 跑一次要多久
    group: str  # 属于哪个互斥回调组
    is_timer: bool = False  # 定时器排在订阅前面先看

**函数 `simulate`**：返回每个回调：跑了几次、漏了几拍、最多晚了多少毫秒才开始。

In [ ]:
def simulate(callbacks: list[Callback], threads: int, total_ms: int = 1000) -> dict:
    """返回每个回调：跑了几次、漏了几拍、最多晚了多少毫秒才开始。"""
    pending: dict[str, float | None] = {c.name: None for c in callbacks}  # 等着跑的那一次是几点到的
    next_release = {c.name: c.period_ms for c in callbacks}
    stats = {c.name: {"runs": 0, "missed": 0, "max_late_ms": 0.0} for c in callbacks}
    busy: list[tuple[int, str]] = []  # (忙到几点, 占着哪个组)
    for t in range(total_ms):
        busy = [(end, g) for end, g in busy if end > t]
        for c in callbacks:
            while next_release[c.name] <= t:
                if c.is_timer and pending[c.name] is not None:
                    stats[c.name]["missed"] += 1  # 上一拍还没跑，这一拍又到了：漏拍（定时器晚了不补发）
                pending[c.name] = next_release[c.name]  # 定时器记最新一拍；图像只留最新一张
                next_release[c.name] += c.period_ms
        for c in sorted(callbacks, key=lambda c: not c.is_timer):  # 定时器排在前面，先看
            if len(busy) >= threads:
                break
            if pending[c.name] is None or any(g == c.group for _, g in busy):
                continue
            late = t - pending[c.name]
            s = stats[c.name]
            s["runs"] += 1
            s["max_late_ms"] = max(s["max_late_ms"], round(late, 1))
            pending[c.name] = None
            busy.append((t + c.cost_ms, c.group))
    return stats

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 · 单线程 executor：一个线程、默认组（互斥），图像回调一跑 40 ms，控制回路只能等")
single = [Callback("control 50 Hz", 20, 1, "default", True), Callback("image 30 Hz", 1000 / 30, 40, "default")]
for name, s in simulate(single, threads=1).items():
    print(f"  {name:<14} 跑了 {s['runs']:>2} 次  漏拍 {s['missed']:>2}  最多晚 {s['max_late_ms']} ms")

In [ ]:
print("\n第二步 · 多线程 executor + 两个互斥组：控制一组、图像一组，两个线程")
grouped = [Callback("control 50 Hz", 20, 1, "control_group", True),
           Callback("image 30 Hz", 1000 / 30, 40, "io_group")]
for name, s in simulate(grouped, threads=2).items():
    print(f"  {name:<14} 跑了 {s['runs']:>2} 次  漏拍 {s['missed']:>2}  最多晚 {s['max_late_ms']} ms")

## 3. 代码里才看得到的

1. **为什么不用真线程**：真 Python 线程受 GIL 和系统调度影响，每次结果不同；按毫秒推进的模拟每次都一样，适合核对。
2. **"漏拍"的定义**：50 Hz 定时器下一拍到点时上一拍还没跑，就记一次漏拍，晚了的那拍不补发——rclpy 的定时器也是这样。
3. **图像只留最新一张**：相当于订阅用 Keep last 1。30 Hz × 40 ms = 每秒要 1.2 秒的活，单线程注定跟不上，旧图被新图顶掉。
4. **1 秒里定时器到点 49 次**（20、40、…、980 ms），所以分组后"跑了 49 次"就是一拍不漏。

## 4. 自己改一改

- 把图像回调的 `cost_ms` 改成 10，再跑单线程，看控制回路还漏不漏拍；
- 把 `threads=2` 改成 `threads=1`（保留两个组），看分组但只有一个线程时会怎样。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **spin**：spin 就是单线程 executor；消息留在中间件，等回调来取。
- **调度**：快时先来后到，慢时轮转；一个慢回调拖累全节点。
- **回调组**：互斥组一个一个来，可重入组随便并行；不同组总能并行；默认组是互斥组。
- **多线程不分组 = 还是串行**；同步等待和被等待的回调放不同组。